# 중복 프로파일 가중치 — 같은 환자를 두 번 세지 않기

train 에는 변이 프로파일이 **완전히 같은** 행들이 있다. 쌍둥이 422 쌍은 전부 (KIPAN, KIRC) 아니면 (GBMLGG, LGG) 이고, 같은 라벨 쌍은 0% 다. 같은 환자를 상위 코호트와 하위 코호트 라벨로 두 번 넣은 데이터 구성의 결과다.

그러면 그 환자는 학습에서 두 번 발언한다. `group_size_inverse_weight` 는 그룹 크기의 역수를 곱해 발언권을 한 번으로 되돌린다.

> 판정은 `oof_macro_f1_singleton`(단독 행)으로 한다. 전체 OOF 는 쌍둥이 행이 섞여 있어 가중치 변경이 자기 자신을 평가하는 꼴이 된다.

In [ ]:
import sys
from pathlib import Path

# notebooks/ 에서 열든 저장소 루트에서 열든 같은 곳을 가리키게 한다.
ROOT = Path.cwd()
while not (ROOT / "src" / "cancer_hack").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "process"
print(ROOT)


In [ ]:
import numpy as np
import pandas as pd

from cancer_hack.models_gbdt import (
    balanced_sample_weight,
    group_size_inverse_weight,
    resolve_sample_weight,
)

folds = pd.read_parquet(DATA_PROC / "train_folds.parquet")
domain = pd.read_parquet(DATA_PROC / "train_domain_features.parquet")
assert (folds["ID"].to_numpy() == domain["ID"].to_numpy()).all()

y = domain["SUBCLASS"].to_numpy()
groups = folds["group_key"].to_numpy()
print(f"train {len(y)}행 · 그룹 {len(set(groups)):,}개 · 클래스 {len(set(y))}개")


## 1. 그룹 크기 분포 — 거의 전부 크기 2 이고 하나만 크게 튄다

In [ ]:
sizes = pd.Series(groups).value_counts()
distribution = sizes.value_counts().sort_index()
distribution.name = "그룹 수"
distribution.index.name = "그룹 크기"
print(distribution.to_dict())

biggest = sizes.idxmax()
members = np.where(groups == biggest)[0]
print(f"\n가장 큰 그룹 {sizes.max()}행 — 라벨 구성:")
print(pd.Series(y[members]).value_counts().head())


## 2. 왜 fold 밖에서 세면 안 되는가

`skf5` 는 같은 그룹을 fold 로 갈라 놓는다. 그 경우 학습에는 한쪽만 들어가므로 그 행의 발언권은 이미 한 번이다. 전역 카운트로 깎으면 두 번 깎는 셈이다.

In [ ]:
crossing = []
for cv_column in ("fold_skf5", "fold_group5"):
    per_group = folds.groupby("group_key")[cv_column].nunique()
    crossing.append({"CV": cv_column, "fold 를 가로지르는 그룹": int((per_group > 1).sum())})
pd.DataFrame(crossing).set_index("CV")


## 3. 가중치 분포와 유효 표본 크기

In [ ]:
def effective_sample_size(weight):
    """(sum w)^2 / sum w^2 — 가중치가 실질적으로 몇 행어치인지."""
    return float(weight.sum() ** 2 / np.square(weight).sum())


rows = []
for name in ("balanced", "group", "group_sqrt", "balanced+group", "balanced+group_sqrt"):
    weight = resolve_sample_weight(name, y, groups)
    rows.append(
        {
            "가중치": name,
            "평균": round(float(weight.mean()), 6),
            "최소": round(float(weight.min()), 4),
            "최대": round(float(weight.max()), 3),
            "유효 표본": round(effective_sample_size(weight), 1),
        }
    )
weight_frame = pd.DataFrame(rows).set_index("가중치")
print(f"가중치 없음 유효 표본 = {len(y)}")
weight_frame


`power=1.0`(`group`)은 가장 큰 그룹을 행당 0.01 수준으로 눌러 그 표본이 사실상 사라진다. `power=0.5`(`group_sqrt`)는 같은 행을 열 배쯤 높게 둔다. 둘 다 남겨 실험으로 고른다.

In [ ]:
raw_product = balanced_sample_weight(y) * group_size_inverse_weight(groups)
print(f"balanced x group 원시 평균 = {raw_product.mean():.4f}  (1 이 아니다)")
print(f"재정규화 후 평균          = {resolve_sample_weight('balanced+group', y, groups).mean():.4f}")
print("\n안 맞추면 실효 학습률이 조용히 달라져 기존 하이퍼파라미터 튜닝값이 의미를 잃는다.")


## 4. 본 실험 — 가중치 x CV 격자

학습은 `scripts/train_gbdt.py` 가 돌린 결과를 읽어 온다. 노트북에서 다시 돌리면 같은 계산을 두 번 하게 되고, 무엇보다 스크립트가 진실의 출처다.

```powershell
$env:PYTHONUTF8 = "1"
$PY = "D:\Code\Final_Hachathon\code\.venv\Scripts\python.exe"
& $PY scripts/train_gbdt.py --model xgb --configs f4r,f4rw,f4rws --cv all `
    --no-submission --tag dup
```

In [ ]:
import json

LOGS = ROOT / "artifacts" / "logs"
records = []
for path in sorted(LOGS.glob("xgb_dup_*_s42.json")):
    result = json.loads(path.read_text(encoding="utf-8"))
    if "oof_macro_f1" not in result:
        continue
    records.append(
        {
            "config": result["config"],
            "CV": result["cv"],
            "가중치": result["sample_weight"],
            "OOF": round(result["oof_macro_f1"], 4),
            "단독 행": round(result["oof_macro_f1_singleton"], 4),
            "Acc": round(result["oof_accuracy"], 4),
        }
    )
grid = pd.DataFrame(records).sort_values(["CV", "단독 행"], ascending=[True, False])
grid


In [ ]:
skf = grid[grid["CV"] == "skf"].set_index("config")
axes = skf[["OOF", "단독 행"]].plot.bar(
    figsize=(7, 3.5), rot=0, title="가중치별 Macro F1 (skf) — 판정은 단독 행"
)
axes.set_ylim(0.40, None)
axes.grid(alpha=0.3)


## 5. 클래스별 델타 — 가설은 코호트 쌍의 혼동이 줄어드는 것이다

쌍둥이가 몰려 있는 네 클래스(KIPAN·KIRC·GBMLGG·LGG)가 어떻게 움직이는지 본다. 감쇠가 효과가 있다면 여기서 먼저 보여야 한다.

In [ ]:
COHORT_PAIRS = ["KIPAN", "KIRC", "GBMLGG", "LGG"]


def per_class(config, cv="skf"):
    matches = list(LOGS.glob(f"xgb_dup_{config}_*_s42.json"))
    for path in matches:
        result = json.loads(path.read_text(encoding="utf-8"))
        if result.get("cv") == cv:
            return pd.Series(result["per_class_f1"])
    raise FileNotFoundError(f"{config}/{cv} 로그가 없다")


baseline = per_class("f4r")
delta = pd.DataFrame(
    {name: per_class(name) - baseline for name in ("f4rw", "f4rws")}
)
axes = delta.loc[COHORT_PAIRS].plot.bar(
    figsize=(7, 3.5), rot=0, title="f4r 대비 클래스별 F1 델타 — 코호트 쌍"
)
axes.axhline(0, color="black", linewidth=0.8)
axes.grid(alpha=0.3)
delta.reindex(delta["f4rw"].abs().sort_values(ascending=False).index).head(10).round(4)


## 결론

위 표에서 `f4r` 대비 **단독 행** 점수가 오른 설정만 남긴다. 전체 OOF 가 떨어지고 단독 행이 오르는 조합이 나오면 그건 쌍둥이 암기가 줄었다는 뜻이라 리더보드에는 유리할 수 있다 — 두 숫자를 같이 기록한다.

점수는 노션 「모델 성능 기록」에 남긴다. 제출 파일은 로컬에만 만들고 DACON 업로드는 사람이 직접 한다.